# Comparaison des modèles d'embeddings

Ce notebook compare deux stratégies d'embeddings pour le système RAG Puls-Events :
- **HuggingFace** : `paraphrase-multilingual-MiniLM-L12-v2` — local, gratuit
- **Mistral** : `mistral-embed` — API payante

Métriques évaluées (Ragas) :
| Métrique | Ce qu'elle mesure |
|---|---|
| `answer_relevancy` | La réponse répond-elle bien à la question ? |
| `faithfulness` | La réponse est-elle fidèle aux documents récupérés ? |
| `context_precision` | Les chunks récupérés sont-ils pertinents (peu de bruit) ? |
| `context_recall` | Toutes les infos nécessaires ont-elles été récupérées ? |

In [7]:
import sys
from pathlib import Path

# Ajouter la racine du projet au path
ROOT = Path(".").resolve().parent
sys.path.insert(0, str(ROOT))

from dotenv import load_dotenv
load_dotenv(ROOT / ".env")

import os
import time
import json
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

from datasets import Dataset
from ragas import evaluate
from ragas.metrics import answer_relevancy, faithfulness, context_precision, context_recall
from ragas.run_config import RunConfig
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from pydantic import SecretStr
from langchain_mistralai import ChatMistralAI
from langchain_huggingface import HuggingFaceEmbeddings

from scripts.build_index import (
    load_events, events_to_documents, split_documents,
    build_faiss_index, save_index, get_embeddings, get_index_dir, DATA_FILE
)
from scripts.rag_chain import load_index, build_chain

ANNOTATIONS_PATH = ROOT / "tests" / "annotated_qa.json"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

CHUNK_SIZE = 500 # Taille des chunks (ex: 300, 500, 700)
CHUNK_OVERLAP = 50
K = 7 # Nombre de chunks récupérés par le retriever

print(f"Config : chunk_size={CHUNK_SIZE}, k={K}")

Config : chunk_size=500, k=7


## 1. Construction des deux index

In [8]:
events = load_events(DATA_FILE)
docs   = events_to_documents(events)
chunks = split_documents(docs, chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)

1000 evenements charges depuis /home/rapha/ai-engineer/poc-rag/data/clean_events.json
1000 documents crees
3499 chunks apres decoupage


In [9]:
print("Construction de l'index HuggingFace...")
t0 = time.time()
index_hf = build_faiss_index(chunks, provider="huggingface")
save_index(index_hf, get_index_dir("huggingface"))
print(f"  Terminé en {time.time() - t0:.1f}s")

Construction de l'index HuggingFace...
Generation des embeddings via huggingface...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1237.70it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Index FAISS construit.
Index sauvegarde dans /home/rapha/ai-engineer/poc-rag/index/faiss_index_huggingface
  Terminé en 18.0s


In [10]:
print("Construction de l'index Mistral...")
t0 = time.time()
index_mistral = build_faiss_index(chunks, provider="mistral")
save_index(index_mistral, get_index_dir("mistral"))
print(f"  Terminé en {time.time() - t0:.1f}s")

Construction de l'index Mistral...
Generation des embeddings via mistral...
Index FAISS construit.
Index sauvegarde dans /home/rapha/ai-engineer/poc-rag/index/faiss_index_mistral
  Terminé en 46.4s


## 2. Génération des réponses RAG

In [11]:
with open(ANNOTATIONS_PATH, encoding="utf-8") as f:
    annotations = json.load(f)
print(f"{len(annotations)} questions chargées.")


def collect_answers(provider: str, annotations: list[dict], k: int = 5, delay: float = 4.0) -> list[dict]:
    index = load_index(provider)
    chain = build_chain(index, k=k)
    retriever = index.as_retriever(search_kwargs={"k": k})
    rows = []
    for i, item in enumerate(annotations, 1):
        if i > 1:
            time.sleep(delay)
        print(f"  [{i}/{len(annotations)}] {item['question'][:55]}...")
        docs = retriever.invoke(item["question"])
        rows.append({
            "question":     item["question"],
            "answer":       chain.invoke(item["question"]),
            "contexts":     [doc.page_content for doc in docs],
            "ground_truth": item["expected_answer"],
        })
    return rows

15 questions chargées.


In [12]:
print("Génération des réponses — HuggingFace...")
t0 = time.time()
rows_hf = collect_answers("huggingface", annotations, k=K)
print(f"  Terminé en {time.time() - t0:.1f}s")

Génération des réponses — HuggingFace...


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1701.55it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


  [1/15] Y a-t-il des ateliers ou formations artistiques en Île-...


HTTPStatusError: Error response 429 while fetching https://api.mistral.ai/v1/chat/completions: {"object":"error","message":"Rate limit exceeded","type":"rate_limited","param":null,"code":"1300","raw_status_code":429}

In [ ]:
print("Génération des réponses — Mistral...")
t0 = time.time()
rows_mistral = collect_answers("mistral", annotations, k=K)
print(f"  Terminé en {time.time() - t0:.1f}s")

## 3. Évaluation Ragas

In [ ]:
api_key = os.getenv("MISTRAL_API_KEY")
ragas_llm = LangchainLLMWrapper(
    ChatMistralAI(model="mistral-small-latest", api_key=SecretStr(api_key), temperature=0.0)
)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2")
)
run_config = RunConfig(timeout=180, max_workers=1, max_retries=5)
metrics = [answer_relevancy, faithfulness, context_precision, context_recall]


def run_ragas(rows: list[dict]) -> pd.DataFrame:
    result = evaluate(
        dataset=Dataset.from_list(rows),
        metrics=metrics,
        llm=ragas_llm,
        embeddings=ragas_embeddings,
        raise_exceptions=False,
        run_config=run_config,
    )
    return result.to_pandas()

In [ ]:
print("Évaluation HuggingFace...")
t0 = time.time()
df_hf = run_ragas(rows_hf)
print(f"  Terminé en {time.time() - t0:.1f}s")

print("Évaluation Mistral...")
t0 = time.time()
df_mistral = run_ragas(rows_mistral)
print(f"  Terminé en {time.time() - t0:.1f}s")

## 4. Comparaison des résultats

In [ ]:
METRICS = ["answer_relevancy", "faithfulness", "context_precision", "context_recall"]

summary = pd.DataFrame({
    "HuggingFace": df_hf[METRICS].mean(),
    "Mistral":     df_mistral[METRICS].mean(),
}).round(3)

summary["Meilleur"] = summary.idxmax(axis=1)
summary["Δ"] = (summary["Mistral"] - summary["HuggingFace"]).round(3)

print("=" * 55)
print("SCORES MOYENS")
print("=" * 55)
print(summary.to_string())
print("=" * 55)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Graphique 1 : scores moyens
ax = axes[0]
x = range(len(METRICS))
width = 0.35
bars_hf  = ax.bar([i - width/2 for i in x], summary["HuggingFace"], width, label="HuggingFace", color="#4C72B0")
bars_mis = ax.bar([i + width/2 for i in x], summary["Mistral"],     width, label="Mistral",     color="#DD8452")
ax.set_xticks(list(x))
ax.set_xticklabels([m.replace("_", "\n") for m in METRICS], fontsize=9)
ax.set_ylim(0, 1.1)
ax.set_title("Scores moyens par métrique")
ax.legend()
ax.bar_label(bars_hf,  fmt="%.3f", fontsize=8)
ax.bar_label(bars_mis, fmt="%.3f", fontsize=8)

# Graphique 2 : différence Mistral - HuggingFace
ax2 = axes[1]
colors = ["#2ca02c" if v >= 0 else "#d62728" for v in summary["Δ"]]
ax2.barh(METRICS, summary["Δ"], color=colors)
ax2.axvline(0, color="black", linewidth=0.8)
ax2.set_title("Δ (Mistral − HuggingFace)\n(vert = Mistral meilleur)")
ax2.set_xlabel("Différence de score")
for i, v in enumerate(summary["Δ"]):
    ax2.text(v + 0.002 if v >= 0 else v - 0.002, i, f"{v:+.3f}",
             va="center", ha="left" if v >= 0 else "right", fontsize=9)

plt.tight_layout()
plt.savefig(RESULTS_DIR / "compare_embeddings.png", bbox_inches="tight")
plt.show()
print(f"Graphique sauvegardé dans results/compare_embeddings.png")

In [ ]:
# Détail par question
detail = pd.DataFrame({
    "question": [r["question"][:50] + "..." for r in rows_hf],
})
for m in METRICS:
    detail[f"{m}_hf"]      = df_hf[m].values.round(3)
    detail[f"{m}_mistral"] = df_mistral[m].values.round(3)

detail

## 5. Conclusion

Choisis le provider gagnant et mets à jour `.env` :

```
EMBEDDING_PROVIDER=huggingface   # ou mistral
```

Puis mets à jour l'import dans `scripts/build_index.py` pour que `get_embeddings()` utilise le bon provider par défaut.